# prototype clock value in jax
Make a working clock-value code with jax and jax-finufft that is fast enough!

## author:
- **David W. Hogg** (NYU) (Flatiron) (MPIA)

## bugs:
- Not well tested.
- Needs a "clock-value threshold" for inclusion in a database.
- Needs to ask whether there are multiple useful independent clocks in the same star, somehow?

## comments:
- Because my Mac has issues with jax, only use jax when we absolutely need it.

In [ ]:
# !pip install lightkurve
# !pip install jax

In [ ]:
from functools import partial
import numpy as np
import jax
import jax.numpy as jnp
import lightkurve as lk
from astropy.timeseries import LombScargle
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
import time

In [ ]:
jax.config.update('jax_platform_name', 'cpu') # because bad Mac behavior?
jax.config.update("jax_enable_x64", True)

In [ ]:
def get_kepler_data(kic_id, exptime='long'):
    """
    ## Inputs:
    `kic_id`: Kepler ID (str)
    `exptime`: default exposure time, 'long'

    ## Outputs:
    Returns a 5-tuple:
    - `times`, `fluxes`, `errors`: light curve data
    - `delta_f`: frequency resolution, 1 / total observation time
    - `sampling_time`: median time between observations (in days)
    """    
    start = time.time()
    print("starting to obtain data for", kic_id)

    try:
        search_result = lk.search_lightcurve(kic_id, mission = 'Kepler', exptime=exptime)
        if len(search_result) < 1:
            msg = f"get_kepler_data(): no results for {kic_id} at this cadence"
            print(msg)
            update_error_message(kic_id, 'Kepler_long', msg)
            return np.nan, np.nan, np.nan, np.nan, np.nan
    except Exception as e:
        print(f"Exception for {kic_id}: get_kepler_data(): lk.search_lightcurve() failed for {kic_id } with {str(e)}")
        return np.nan, np.nan, np.nan, np.nan, np.nan

    try:
        lc_collection = search_result.download_all()
    except lk.LightkurveError as e:
        print(f"LightkurveError for {kic_id}: get_kepler_data(): search_result.download_all() failed for {kic_id} with {str(e)}")
        return np.nan, np.nan, np.nan, np.nan, np.nan
    except Exception as e:
        print(f"Exception for {kic_id}: get_kepler_data(): search_result.download_all() failed for {kic_id} with {str(e)}")
        return np.nan, np.nan, np.nan, np.nan, np.nan

    try:
        lc = lc_collection.stitch()
        #print("get_kepler_data(): minimum time value", np.min(lc.time.value), np.min(lc.time), lc.time)
    except lk.LightkurveError as e:
        print(f"LightkurveError for {kic_id}: get_kepler_data(): lc_collection.stitch() failed for {kic_id} with {str(e)}")
        return np.nan, np.nan, np.nan, np.nan, np.nan
    except Exception as e:
        print(f"Exception for {kic_id}: get_kepler_data(): lc_collection.stitch() failed for {kic_id} with {str(e)}")
        return np.nan, np.nan, np.nan, np.nan, np.nan

    # unpack, remove bad data, and reorder
    times, fluxes, errors = lc.time.value, lc.flux.value, lc.flux_err.value
    good = np.isfinite(times) & np.isfinite(fluxes) & np.isfinite(errors)
    times, fluxes, errors = times[good], fluxes[good], errors[good]
    idx = np.argsort(times)
    times, fluxes, errors = times[idx], fluxes[idx], errors[idx]

    delta_f = (1/(times[-1] - times[0]))
    sampling_time= np.median(np.diff(times))
    print("get_kepler_data() took", time.time() - start, "seconds")

    return times, fluxes, errors, delta_f, sampling_time

In [ ]:
# get data on a good example

# kicid = "KIC005285607"
kicid = "KIC002162994"
ts, ys, errs, deltaf, deltat = get_kepler_data(kicid)
print(ts.shape, ys.shape, errs.shape, deltaf, deltat)

In [ ]:
# check the data

f = plt.figure(figsize=(9, 3))
plt.scatter(ts, ys, s=1, c="k", marker=".")
plt.title(kicid)

In [ ]:
# get a set of hypotheses for this star

def get_candidate_frequencies(ts, ys, errs, df, dt, max_peaks=32, nterms=8):
    """
    # inputs:
    - `ts`, `ys`, `errs`: the light curve
    - `df`, `dt`: the smallest frequency and the smallest time of relevance

    # bugs:
    - MAGIC minimum frequency of 0.05 made up by Hogg (inverse 20 days)
    - MAGIC oversampling by a factor of either 2 or 4 (I don't know which)
    """
    fs = np.arange(0.05, 0.5 / dt, 0.25 * df)
    ps = LombScargle(ts, ys, errs, nterms=nterms).power(fs)
    idxs, _ = find_peaks(ps, distance=4)
    ii = np.argsort(ps[idxs])[::-1]
    idxs = idxs[ii]
    if len(idxs) > max_peaks:
        idxs = idxs[:max_peaks]
    return fs[idxs]

In [ ]:
candidate_fs = get_candidate_frequencies(ts, ys, errs, deltaf, deltat)
candidate_oms = 2. * np.pi * candidate_fs
print(candidate_oms)

In [ ]:
# design matrix for an M-component Fourier series
# Note: solving this problem with standard (non-finufft) methods

@partial(jax.jit, static_argnums=2)
def design_matrix(om, t, M):
    ms1, ms2 = jnp.arange(M + 1), jnp.arange(1, M + 1)
    return jnp.concat((jnp.cos(ms1[None, :] * om * t[:, None]),
                       jnp.sin(ms2[None, :] * om * t[:, None])), axis=1), \
           jnp.concat((ms1, ms2))                    

In [ ]:
# time to get the clock value

@partial(jax.jit, static_argnums=4)
def fourier_wls_fit(om, t, y, iv, M):
    X, m = design_matrix(om, t, M)
    return X, m, jnp.linalg.solve(X.T @ (iv[:, None] * X),
                                  X.T @ (iv * ys))

@partial(jax.jit, static_argnums=4)
def clock_value(om, t, y, iv, M):
    """
    # inputs:
    - `om`: frequency to test
    - `t`, `y`, `iv`: light curve (iv is inverse variance, not error)
    - `M`: degree of Fourier series

    # bugs:
    - This is very affected by outliers; need to remove those somehow. But *don't* use `jnp.median()`!
    - Maybe we should use IRLS to do the fit.
    - Maybe we should penalize (or increase) MSE according to model complexity `2 * M + 1`.
    """
    X, m, pars = fourier_wls_fit(om, t, y, iv, M)
    mse = jnp.sum(iv * (y - X @ pars) ** 2) / jnp.sum(iv) # weighted mean
    return (om ** 2 / mse) * jnp.sum(m ** 2 * pars ** 2)

clock_values = jax.vmap(clock_value, in_axes=(0, None, None, None, None))

In [ ]:
# get best clock-value frequency near a peak
# Note: this is the parabola trick in the log

def get_best_clock(om0, t, y, iv, Mmax, df, dt):
    """
    # inputs:
    - `om0`: first guess at a good clock (angular) frequency omega
    - `t`, `y`, `iv`: the light curve
    - `Mmax`: the maximum degree of the Fourier series (not necessarily the degree)
    - `df`: the frequency resolution (non-angular frequency) in the data
    - `dt`: the sampling time (related to Nyquist).

    # bugs:
    - Takes one input as angular frequency and another as frequency.
    - MAGIC 0.05

    # notes:
    - Recursion is insane.
    - Works in the log for stability.
    """
    nyquist = np.pi / dt # angular-frequency units
    M = max(1, min(Mmax, int(nyquist // om0)))
    do = 0.05 * np.pi * df # magic 0.05
    oms = np.array([om0 - do, om0, om0 + do])
    ys = np.log(clock_values(oms, t, y, iv, M))
    if np.argmax(ys) != 1:
        return get_best_clock(oms[np.argmax(ys)], t, y, iv, M, df, dt)
    ii = jnp.argmax(ys)
    foo = jnp.polyfit(oms, ys, 2)
    om = jnp.roots(jnp.polyder(foo), strip_zeros=False).real
    return om[0], jnp.exp(jnp.polyval(foo, om))[0], M

In [ ]:
# search all candidate omegas, using a dumb loop

Mmax = 128
ivars = 1. / errs ** 2
oms, values = np.zeros_like(candidate_oms), np.zeros_like(candidate_oms)
Ms = np.zeros_like(candidate_oms).astype(int)
for i, om0 in enumerate(candidate_oms):
    oms[i], values[i], Ms[i] = get_best_clock(om0, ts, ys, ivars, Mmax, deltaf, deltat)
idx = np.argmax(values)
best_omega, best_value, best_M = oms[idx], values[idx], Ms[idx]
print("the best clock is", best_omega, best_value)

In [ ]:
# plot the clocks we tested in this star

plt.scatter(best_omega, best_value, marker="x", s=80, c="r")
plt.scatter(oms, values, marker="o", c="k", s=3. * np.sqrt(Ms))
# plt.semilogx()
plt.loglog()
plt.title(kicid)
# plt.xlim(1.6, 1.62)

In [ ]:
# look at best frequency

best_period = 2. * jnp.pi / best_omega
f = plt.figure(figsize=(9, 3))
_, _, pars = fourier_wls_fit(best_omega, ts, ys, ivars, best_M)
thetas_plot = np.linspace(0., 4. * np.pi, 1000)
X_plot, _ = design_matrix(best_omega, thetas_plot / best_omega, best_M)
plt.scatter((best_omega * ts) % (2. * np.pi), ys, s=1, c="k", marker=".")
plt.scatter((best_omega * ts) % (2. * np.pi) + 2. * np.pi, ys, s=1, c="k", marker=".")
plt.plot(thetas_plot, X_plot @ pars, "r-")
plt.title(f"{kicid} at period {best_period:.6f} d")
plt.xlim(0., 4. * np.pi)
plt.xlabel("phase [rad]")